# 01 - Time Series EDA

**Project:** Demand Forecasting for Operations

**Dataset:** Synthetic hourly operational demand, monthly budget, and weekly workforce metrics based on BLS/BTS patterns.

**Objective:** Understand seasonal patterns, trends, and anomalies in operational demand.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries loaded.")

In [ ]:
DATA_DIR = '../data'

hourly = pd.read_csv(f'{DATA_DIR}/hourly_demand.csv', parse_dates=['datetime'])
monthly = pd.read_csv(f'{DATA_DIR}/monthly_budget.csv')
weekly = pd.read_csv(f'{DATA_DIR}/weekly_metrics.csv', parse_dates=['week_start'])

print(f"Hourly records: {len(hourly):,}")
print(f"Monthly records: {len(monthly)}")
print(f"Weekly records: {len(weekly)}")
print(f"\nHourly date range: {hourly['datetime'].min()} to {hourly['datetime'].max()}")

## Hourly Demand Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Overall demand over time
hourly.set_index('datetime')['demand'].plot(ax=axes[0,0], color='steelblue', alpha=0.7)
axes[0,0].set_title('Hourly Demand Over 2 Years')
axes[0,0].set_ylabel('Demand')

# Hourly pattern
hourly.groupby('hour')['demand'].mean().plot(ax=axes[0,1], color='coral', marker='o')
axes[0,1].set_title('Average Demand by Hour of Day')
axes[0,1].set_xlabel('Hour')
axes[0,1].set_ylabel('Avg Demand')

# Day of week pattern
dow_map = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
hourly['dow_name'] = hourly['day_of_week'].map(dow_map)
hourly.groupby('dow_name')['demand'].mean().reindex(
    ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
).plot(ax=axes[1,0], color='seagreen', marker='o')
axes[1,0].set_title('Average Demand by Day of Week')
axes[1,0].set_ylabel('Avg Demand')

# Monthly seasonality
hourly.groupby('month')['demand'].mean().plot(ax=axes[1,1], color='purple', marker='o')
axes[1,1].set_title('Average Demand by Month')
axes[1,1].set_xlabel('Month')
axes[1,1].set_ylabel('Avg Demand')

plt.tight_layout()
plt.savefig('../figures/demand_01_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/demand_01_patterns.png")

## Workforce Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

weekly.set_index('week_start')['headcount'].plot(ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Headcount Over Time')

weekly.set_index('week_start')['turnover_rate'].plot(ax=axes[0,1], color='coral')
axes[0,1].set_title('Turnover Rate Over Time')
axes[0,1].set_ylabel('Turnover Rate')

weekly.set_index('week_start')['overtime_ratio'].plot(ax=axes[1,0], color='seagreen')
axes[1,0].set_title('Overtime Ratio Over Time')
axes[1,0].set_ylabel('Overtime Ratio')

weekly.set_index('week_start')['productivity_index'].plot(ax=axes[1,1], color='purple')
axes[1,1].set_title('Productivity Index Over Time')

plt.tight_layout()
plt.savefig('../figures/demand_01_workforce.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/demand_01_workforce.png")

## Budget Variance Analysis

In [ ]:
# Budget vs Actual by category
pivot = monthly.pivot(index='month', columns='category', values=['budget_allocated', 'actual_spent'])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

cats = monthly['category'].unique()
for idx, cat in enumerate(cats):
    cat_data = monthly[monthly['category'] == cat]
    ax = axes[idx // 3, idx % 3]
    ax.plot(cat_data['month'], cat_data['budget_allocated'], label='Budget', marker='o')
    ax.plot(cat_data['month'], cat_data['actual_spent'], label='Actual', marker='s')
    ax.set_title(f'{cat}')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()

plt.tight_layout()
plt.savefig('../figures/demand_01_budget.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/demand_01_budget.png")

## Key Findings

1. **Hourly Pattern:** Clear rush-hour peaks at 7-9 AM and 5-7 PM
2. **Weekly Pattern:** Weekdays 30% higher demand than weekends
3. **Seasonal:** Summer months show 15% higher demand
4. **Overtime:** Correlates with demand spikes; peaks in Q4
5. **Budget:** Labor is largest category; most categories within 8% variance

Next: ARIMA and Prophet Models (Notebook 02)